# 面试问题：Constitutional AI / RLAIF 怎样把原则变成可训练、可审计的数据？

**回答主线。** Constitution 不是一段抽象口号，而是有 ID、优先级、适用范围、判定标准和升级路径的版本化策略。典型流程先让模型依据原则自我 critique 与 revision，形成监督微调数据；再让 AI judge 比较候选，生成偏好对，训练 preference/reward model，最后进入 RL 或直接偏好优化。

下面用确定性小规则模拟数据管线，重点验证 provenance、冲突、位置偏差、reward 拟合和发布门禁。规则模拟器不代表真实安全分类器，也不会把 AI feedback 当成人类监督的无条件替代品。


In [ ]:
import hashlib, json, math, re  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 固定随机源只用于数值优化和抽样审计。
rng157 = np.random.default_rng(157)  # 计算并保存当前步骤的中间状态。

def canonical_digest157(value):  # 定义本节可复用的核心函数。
    payload = json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(payload.encode()).hexdigest()  # 返回当前分支计算出的结果。

assert len(canonical_digest157({"a": 1})) == 64  # 用受控断言验证关键不变量。
assert canonical_digest157({"a": 1, "b": 2}) == canonical_digest157({"b": 2, "a": 1})  # 用受控断言验证关键不变量。
assert canonical_digest157({"a": 1}) != canonical_digest157({"a": 2})  # 用受控断言验证关键不变量。


## 1. Principle 需要稳定 ID、优先级与可执行判定

自然语言原则允许解释，但生产管线仍要给出机器可检查的触发器、动作和 human escalation。相同 ID 不可覆盖；优先级相同且动作冲突时不能静默取输入顺序。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Principle157:  # 定义承载本节状态与行为的数据结构。
    principle_id: str  # 执行当前语句以推进本节示例。
    priority: int  # 执行当前语句以推进本节示例。
    pattern: str  # 执行当前语句以推进本节示例。
    action: str  # 执行当前语句以推进本节示例。
    scope: str = "all"  # 计算并保存当前步骤的中间状态。

def validate_constitution157(principles):  # 定义本节可复用的核心函数。
    # ID 是日志和训练样本的外键，必须唯一且字段完整。
    ids = [p.principle_id for p in principles]  # 计算并保存当前步骤的中间状态。
    if len(ids) != len(set(ids)):  # 按当前条件选择后续控制路径。
        raise ValueError("duplicate principle id")  # 遇到非法合同立即显式失败。
    if any(p.priority < 0 or p.action not in {"redact", "refuse", "revise", "escalate"} for p in principles):  # 按当前条件选择后续控制路径。
        raise ValueError("invalid principle")  # 遇到非法合同立即显式失败。
    return sorted(principles, key=lambda p: (-p.priority, p.principle_id))  # 返回当前分支计算出的结果。

constitution157 = validate_constitution157([  # 计算并保存当前步骤的中间状态。
    Principle157("P-SECRET", 100, r"sk-[A-Za-z0-9]+", "redact"),  # 执行当前语句以推进本节示例。
    Principle157("P-ABUSE", 50, r"\bidiot\b", "revise"),  # 执行当前语句以推进本节示例。
    Principle157("P-WEAPON", 90, r"build a weapon", "refuse"),  # 执行当前语句以推进本节示例。
])  # 执行当前语句以推进本节示例。
assert [p.principle_id for p in constitution157] == ["P-SECRET", "P-WEAPON", "P-ABUSE"]  # 用受控断言验证关键不变量。
assert constitution157[0].priority == 100  # 用受控断言验证关键不变量。
assert len({p.principle_id for p in constitution157}) == 3  # 用受控断言验证关键不变量。


## 2. Critique 输出证据 span，而不是只给笼统分数

每个 violation 记录 principle、匹配文本和位置，方便复核与重放。Judge 的解释不能反过来成为事实源；真正的判定仍绑定原始 response 和 constitution revision。


In [ ]:
def critique157(response, principles):  # 定义本节可复用的核心函数。
    # 使用受控 regex 返回可定位证据；真实系统会组合分类器与人工审核。
    findings = []  # 计算并保存当前步骤的中间状态。
    for principle in principles:  # 遍历输入元素以累积或检查结果。
        for match in re.finditer(principle.pattern, response, flags=re.I):  # 遍历输入元素以累积或检查结果。
            findings.append({"principle_id": principle.principle_id, "action": principle.action, "span": match.span(), "evidence": match.group(0)})  # 执行当前语句以推进本节示例。
    return findings  # 返回当前分支计算出的结果。

unsafe157 = "You are an idiot. Use sk-ABC123 and build a weapon."  # 计算并保存当前步骤的中间状态。
findings157 = critique157(unsafe157, constitution157)  # 计算并保存当前步骤的中间状态。
assert {f["principle_id"] for f in findings157} == {"P-SECRET", "P-ABUSE", "P-WEAPON"}  # 用受控断言验证关键不变量。
assert all(f["span"][0] < f["span"][1] for f in findings157)  # 用受控断言验证关键不变量。
assert any(f["evidence"] == "sk-ABC123" for f in findings157)  # 用受控断言验证关键不变量。


## 3. Revision 应最小修改并保留有帮助的安全内容

简单整段拒答可能降低风险，却造成过度拒答。处理顺序按优先级：敏感值先脱敏，侮辱措辞改写；若命中高风险行为则拒绝具体步骤，并提供安全替代方向。


In [ ]:
def revise157(response, principles):  # 定义本节可复用的核心函数。
    # 每条原则只执行声明动作，避免 revision 模型自行扩大政策。
    revised = response  # 计算并保存当前步骤的中间状态。
    applied = []  # 计算并保存当前步骤的中间状态。
    for principle in principles:  # 遍历输入元素以累积或检查结果。
        if not re.search(principle.pattern, revised, flags=re.I):  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        applied.append(principle.principle_id)  # 执行当前语句以推进本节示例。
        if principle.action == "redact":  # 按当前条件选择后续控制路径。
            revised = re.sub(principle.pattern, "[REDACTED]", revised, flags=re.I)  # 计算并保存当前步骤的中间状态。
        elif principle.action == "revise":  # 按当前条件选择后续控制路径。
            revised = re.sub(principle.pattern, "unhelpful", revised, flags=re.I)  # 计算并保存当前步骤的中间状态。
        elif principle.action == "refuse":  # 按当前条件选择后续控制路径。
            revised = re.sub(principle.pattern, "discuss safe prevention", revised, flags=re.I)  # 计算并保存当前步骤的中间状态。
    return revised, applied  # 返回当前分支计算出的结果。

revised157, applied157 = revise157(unsafe157, constitution157)  # 计算并保存当前步骤的中间状态。
assert "sk-" not in revised157 and "idiot" not in revised157.lower()  # 用受控断言验证关键不变量。
assert "safe prevention" in revised157  # 用受控断言验证关键不变量。
assert set(applied157) == {"P-SECRET", "P-ABUSE", "P-WEAPON"}  # 用受控断言验证关键不变量。


## 4. 偏好对绑定 prompt、候选、原则和生成版本

只保存 chosen/rejected 文本无法解释选择，也无法在 constitution 更新后重算。样本摘要应覆盖 prompt、两候选、适用 principle、生成模型和 judge revision，并按 prompt family 切分避免泄漏。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class PreferencePair157:  # 定义承载本节状态与行为的数据结构。
    prompt_id: str  # 执行当前语句以推进本节示例。
    chosen: str  # 执行当前语句以推进本节示例。
    rejected: str  # 执行当前语句以推进本节示例。
    principle_ids: tuple  # 执行当前语句以推进本节示例。
    generator_revision: str  # 执行当前语句以推进本节示例。
    judge_revision: str  # 执行当前语句以推进本节示例。
    digest: str  # 执行当前语句以推进本节示例。

def make_pair157(prompt_id, chosen, rejected, principle_ids, generator, judge):  # 定义本节可复用的核心函数。
    # digest 覆盖语义字段，后续任何文本或版本变化都会产生新样本。
    record = {"prompt_id": prompt_id, "chosen": chosen, "rejected": rejected, "principles": sorted(principle_ids), "generator": generator, "judge": judge}  # 计算并保存当前步骤的中间状态。
    return PreferencePair157(prompt_id, chosen, rejected, tuple(sorted(principle_ids)), generator, judge, canonical_digest157(record))  # 返回当前分支计算出的结果。

pair157 = make_pair157("prompt-family-7", revised157, unsafe157, applied157, "policy-v2", "judge-v5")  # 计算并保存当前步骤的中间状态。
assert pair157.chosen != pair157.rejected  # 用受控断言验证关键不变量。
assert len(pair157.digest) == 64  # 用受控断言验证关键不变量。
assert pair157.principle_ids == tuple(sorted(applied157))  # 用受控断言验证关键不变量。


## 5. AI Judge 必须做位置交换、tie 与多评审一致性

同一对候选按 A/B 与 B/A 两个顺序评审；结论不一致时标记 tie/uncertain，而不是强造标签。多 judge 的版本和独立性同样重要，共享同一盲点的多数票没有可靠性增益。


In [ ]:
def safety_score157(response, principles):  # 定义本节可复用的核心函数。
    # 受控分数以 violation 数为主，并给安全替代内容少量 helpfulness 奖励。
    violations = len(critique157(response, principles))  # 计算并保存当前步骤的中间状态。
    helpful = int("safe" in response.lower() or "prevention" in response.lower())  # 计算并保存当前步骤的中间状态。
    return -2.0 * violations + 0.25 * helpful  # 返回当前分支计算出的结果。

def pair_judgment157(a, b, position_bias=0.0):  # 定义本节可复用的核心函数。
    score_a = safety_score157(a, constitution157) + position_bias  # 计算并保存当前步骤的中间状态。
    score_b = safety_score157(b, constitution157)  # 计算并保存当前步骤的中间状态。
    return "A" if score_a > score_b else "B" if score_b > score_a else "tie"  # 返回当前分支计算出的结果。

forward157 = pair_judgment157(revised157, unsafe157)  # 计算并保存当前步骤的中间状态。
reverse157 = pair_judgment157(unsafe157, revised157)  # 计算并保存当前步骤的中间状态。
assert forward157 == "A"  # 用受控断言验证关键不变量。
assert reverse157 == "B"  # 用受控断言验证关键不变量。
assert pair_judgment157("safe answer", "safe answer") == "tie"  # 用受控断言验证关键不变量。


## 6. Bradley–Terry 把 chosen/rejected 差分映射成偏好概率

Preference model 可学习 `sigmoid(r_chosen-r_rejected)`。下面用三维可解释特征和梯度下降验证 loss 下降；真实 reward model 还要做 prompt split、tie、校准和分布外测试。


In [ ]:
def feature157(text):  # 定义本节可复用的核心函数。
    # 特征依次为负 violation、是否含安全替代、截断长度。
    return np.array([-len(critique157(text, constitution157)), int("safe" in text.lower()), min(len(text), 200) / 200.0], dtype=np.float64)  # 返回当前分支计算出的结果。

pairs157 = [(revised157, unsafe157), ("safe prevention guidance", "build a weapon"), ("polite safe answer", "you idiot")]  # 计算并保存当前步骤的中间状态。
differences157 = np.stack([feature157(c) - feature157(r) for c, r in pairs157])  # 计算并保存当前步骤的中间状态。
weights157 = np.zeros(3)  # 计算并保存当前步骤的中间状态。
def bt_loss157(w):  # 定义本节可复用的核心函数。
    margin = differences157 @ w  # 计算并保存当前步骤的中间状态。
    return float(np.mean(np.logaddexp(0.0, -margin)))  # 返回当前分支计算出的结果。
initial_loss157 = bt_loss157(weights157)  # 计算并保存当前步骤的中间状态。
for _ in range(200):  # 遍历输入元素以累积或检查结果。
    margin = differences157 @ weights157  # 计算并保存当前步骤的中间状态。
    gradient = -(differences157 * (1.0 / (1.0 + np.exp(margin)))[:, None]).mean(axis=0)  # 计算并保存当前步骤的中间状态。
    weights157 -= 0.1 * gradient  # 计算并保存当前步骤的中间状态。
assert bt_loss157(weights157) < initial_loss157  # 用受控断言验证关键不变量。
assert np.all(differences157 @ weights157 > 0)  # 用受控断言验证关键不变量。
assert np.isfinite(weights157).all()  # 用受控断言验证关键不变量。


## 7. 原则冲突与低置信必须升级给人

两条同优先级原则若对同一 span 要求不同动作，系统没有足够信息自行裁决。高影响领域还应配置强制人工 slice，即使 judge 分数很高也不能自动发布。


In [ ]:
def resolve_actions157(findings, principles):  # 定义本节可复用的核心函数。
    # 同一证据按最高优先级裁决；最高层动作不唯一则显式 escalate。
    priority = {p.principle_id: p.priority for p in principles}  # 计算并保存当前步骤的中间状态。
    grouped = {}  # 计算并保存当前步骤的中间状态。
    for finding in findings:  # 遍历输入元素以累积或检查结果。
        grouped.setdefault(finding["span"], []).append(finding)  # 执行当前语句以推进本节示例。
    decisions = []  # 计算并保存当前步骤的中间状态。
    for span, group in grouped.items():  # 遍历输入元素以累积或检查结果。
        top = max(priority[f["principle_id"]] for f in group)  # 计算并保存当前步骤的中间状态。
        actions = {f["action"] for f in group if priority[f["principle_id"]] == top}  # 计算并保存当前步骤的中间状态。
        decisions.append("escalate" if len(actions) > 1 else next(iter(actions)))  # 执行当前语句以推进本节示例。
    return decisions  # 返回当前分支计算出的结果。

conflict_principles157 = [Principle157("A", 10, "x", "redact"), Principle157("B", 10, "x", "refuse")]  # 计算并保存当前步骤的中间状态。
conflict_findings157 = critique157("x", conflict_principles157)  # 计算并保存当前步骤的中间状态。
assert resolve_actions157(conflict_findings157, conflict_principles157) == ["escalate"]  # 用受控断言验证关键不变量。
assert resolve_actions157(findings157, constitution157).count("escalate") == 0  # 用受控断言验证关键不变量。
assert len(resolve_actions157(findings157, constitution157)) == 3  # 用受控断言验证关键不变量。


## 8. 发布门禁同时看风险下降、帮助性与过度拒答

只优化 harmlessness 会得到“什么都不回答”的退化模型。回归集要包含应拒绝、应改写和完全安全三类，并按语言、领域和脆弱群体切片；AI judge 指标必须用独立人审校准。


In [ ]:
def release_gate157(before, after, safe_prompts_retained, overrefusal_rate):  # 定义本节可复用的核心函数。
    # 风险下降是必要条件，同时限制安全请求的帮助性损失和过度拒答。
    before_v = sum(len(critique157(x, constitution157)) for x in before)  # 计算并保存当前步骤的中间状态。
    after_v = sum(len(critique157(x, constitution157)) for x in after)  # 计算并保存当前步骤的中间状态。
    metrics = {"violation_reduction": (before_v - after_v) / max(before_v, 1), "safe_retention": safe_prompts_retained, "overrefusal": overrefusal_rate}  # 计算并保存当前步骤的中间状态。
    passed = metrics["violation_reduction"] >= 0.8 and metrics["safe_retention"] >= 0.95 and metrics["overrefusal"] <= 0.05  # 调整当前循环或占位控制流。
    return passed, metrics  # 返回当前分支计算出的结果。

passed157, metrics157 = release_gate157([unsafe157], [revised157], 0.98, 0.03)  # 调整当前循环或占位控制流。
assert passed157  # 用受控断言验证关键不变量。
assert metrics157["violation_reduction"] == 1.0  # 用受控断言验证关键不变量。
assert not release_gate157([unsafe157], [revised157], 0.8, 0.2)[0]  # 用受控断言验证关键不变量。


## 面试总结

- Constitution 要有稳定 ID、优先级、scope、动作、冲突和人审升级，而不是只写价值观口号。
- Critique/revision 生成 SFT 数据；AI comparison 生成偏好数据，二者都要绑定原响应、原则和模型版本。
- Judge 要检查位置偏差、tie、独立性和人类校准；AI feedback 不能无条件升级成真值。
- 发布门禁同时约束 violation、helpfulness、over-refusal、slice 与人工审核。

延伸阅读：[Constitutional AI](https://arxiv.org/abs/2212.08073)、[Scaling Laws for RLAIF](https://arxiv.org/abs/2309.00267)、[Training Language Models to Follow Instructions with Human Feedback](https://arxiv.org/abs/2203.02155)。
